In [1]:
import os
import torch
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [2]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Perangkat yang digunakan: {device}")

Perangkat yang digunakan: cuda


In [3]:
# Load Document
file_path = r"D:\Project AI\Generative AI LLM\buku_panduan_gen_ai.pdf"
loader = PyMuPDFLoader(file_path)
documents = loader.load()

documents[8]

Document(metadata={'producer': 'iLovePDF', 'creator': 'Adobe InDesign 20.3 (Windows)', 'creationdate': '2025-06-03T13:36:46+07:00', 'source': 'D:\\Project AI\\Generative AI LLM\\buku_panduan_gen_ai.pdf', 'file_path': 'D:\\Project AI\\Generative AI LLM\\buku_panduan_gen_ai.pdf', 'total_pages': 42, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-06-04T05:01:35+00:00', 'trapped': '', 'modDate': 'D:20250604050135Z', 'creationDate': "D:20250603133646+07'00'", 'page': 8}, page_content='8\nBAB II: Pemahaman Teknologi Generative AI\n2.1. Perbedaan AI dan Generative AI\nSebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk \nmemahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-\ncara khusus.\n1.\t Artificial Intelligence (AI)\n•\t AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-\nagai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.\n•\t Cont

In [4]:
# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", "."]
)

chunks = text_splitter.split_documents(documents)
print(f"Dokumen dipecah menjadi {len(chunks)} chunks")

Dokumen dipecah menjadi 78 chunks


In [5]:
# Storing / Embedding
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': device}
)

In [6]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [8]:
query = "Apa itu Generative AI"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"===== Dokumen {i} ======")
    print(doc.page_content)
    print()

===== Dokumen 1 ======
8
BAB II: Pemahaman Teknologi Generative AI
2.1. Perbedaan AI dan Generative AI
Sebelum memahami cara kerja Generative AI (GenAI), penting bagi mahasiswa untuk 
memahami perbedaan antara Artificial Intelligence (AI) secara umum dan GenAI se-
cara khusus.
1.	 Artificial Intelligence (AI)
•	 AI adalah teknologi yang dirancang untuk meniru kecerdasan manusia dalam berb-
agai tugas seperti analisis data, pengenalan pola, dan pengambilan keputusan.
•	 Contoh AI meliputi asisten virtual seperti Siri dan Google Assistant, algoritma pen-
carian Google, dan sistem rekomendasi Netflix atau Spotify.
•	 AI umumnya bersifat berbasis aturan atau pembelajaran mesin yang membantu ma-
nusia dalam mengambil keputusan.
2. Generative AI (GenAI)
•	 GenAI adalah cabang dari AI yang mampu menghasilkan konten baru, seperti teks, 
gambar, video, dan suara, berdasarkan data yang telah dipelajarinya.
•	 Model GenAI menggunakan teknik deep learning dan neural networks untuk men-

===== Doku

In [9]:
# konfigurasi Kuantisasi 4-Bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [13]:
model_name = "unsloth/Llama-3.2-3B-Instruct"

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="cuda:0"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

d:\Project AI\Generative AI LLM\venv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [14]:
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=128
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

Device set to use cuda:0


In [15]:
# Llama 3 menggunakan format <|start_header_id|>system<|end_header_id|> dst.
template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
 
Anda adalah asisten AI yang bertugas membantu pengguna. 
Gunakan hanya informasi yang tersedia pada konteks berikut untuk menjawab pertanyaan. 
Jika jawaban tidak ditemukan dalam konteks tersebut, sampaikan dengan jujur bahwa Anda tidak mengetahui jawabannya dan jangan membuat asumsi atau jawaban tambahan. 
Berikan jawaban secara singkat dan jelas.
 
Context:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
 
{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "query"]
)

In [16]:
rag_chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [17]:
def ask_question(query):
    print(f"Pertanyaan: {query}")

    # Jalankan Chain
    response = rag_chain.invoke(query)

    print("Jawaban:")
    print(response)

    # Tampilkan sumber referensi
    docs = retriever.invoke(query)
    print("\nSumber Referensi:")
    for i, doc in enumerate(docs):
        print(f"{i+1}. Halaman {doc.metadata.get('page', '?')}")

In [18]:
query = "apa itu generative ai"
ask_question(query)

Pertanyaan: apa itu generative ai


d:\Project AI\Generative AI LLM\venv\lib\site-packages\bitsandbytes\backends\cuda\ops.py:464: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Jawaban:
Generative AI (Generative Artificial Intelligence) adalah cabang dari Artificial Intelligence (AI) yang dirancang untuk menghasilkan konten baru, seperti teks, gambar, suara, dan video, berdasarkan data yang telah dipelajarinya.

Dalam bahasa lain, Generative AI dapat diartikan sebagai "AI yang bisa menciptakan" atau "AI yang bisa menghasilkan". Ini karena model Generative AI menggunakan teknik deep learning dan neural networks untuk meniru pola dan struktur data, sehingga dapat menghasilkan konten baru yang memiliki kesamaan dengan contoh-contoh

Sumber Referensi:
1. Halaman 8
2. Halaman 8
3. Halaman 38
